## Parameter

In [ ]:
%%writefile parameters.py
"""
Parameters for the compressible 2D Euler solver.
"""

import numpy as np

class Parameters:
    def __init__(self):
        # Gas properties
        self.gamma = 1.4
        self.R = 287.0

        # Grid parameters
        self.Nx = 500  # Number of cells in x-direction
        self.Ny = 100    # Number of cells in y-direction (thin for quasi-1D)
        self.Lx = 10.0  # Domain length in x
        self.Ly = 1 # Domain height in y

        # Time parameters
        self.CFL = 0.4
        self.t_final = 1.0  # Final simulation time
        self.t_print = 0.05  # Save interval

        # Output
        self.output_dir = "output"

        # Derived quantities
        self.update_grid()

    def update_grid(self):
        """Update derived quantities"""
        self.dx = self.Lx / self.Nx
        self.dy = self.Ly / self.Ny

# Global parameters
params = Parameters()

Overwriting parameters.py


## Meshing

In [ ]:
%%writefile mesh.py
"""
Mesh and grid generation.
"""

import numpy as np
from parameters import params

class Mesh:
    def __init__(self):
        self.Nx = params.Nx
        self.Ny = params.Ny
        self.Lx = params.Lx
        self.Ly = params.Ly
        self.dx = params.dx
        self.dy = params.dy

        # Cell centers
        self.x_centers = (np.arange(self.Nx) + 0.5) * self.dx
        self.y_centers = (np.arange(self.Ny) + 0.5) * self.dy
        self.X, self.Y = np.meshgrid(self.x_centers, self.y_centers, indexing='ij')

        print(f"Grid: {self.Nx} x {self.Ny}")
        print(f"Domain: x∈[0,{self.Lx}], y∈[0,{self.Ly}]")
        print(f"dx={self.dx:.6f}, dy={self.dy:.6f}")

# Global mesh object
mesh = Mesh()

Overwriting mesh.py


## state

In [ ]:
%%writefile state.py
"""
Compressible flow state variables.
"""

import numpy as np
from parameters import params
import mesh

class FlowState:
    __slots__ = ['gamma', 'Nx', 'Ny', 'Q', 'rho', 'u', 'v', 'p']

    def __init__(self):
        self.gamma = params.gamma
        self.Nx = mesh.mesh.Nx
        self.Ny = mesh.mesh.Ny

        # Conservative variables Q = [rho, rho*u, rho*v, E]
        self.Q = np.zeros((4, self.Nx, self.Ny))

        # Primitive variables
        self.rho = np.ones((self.Nx, self.Ny))
        self.u = np.zeros((self.Nx, self.Ny))
        self.v = np.zeros((self.Nx, self.Ny))
        self.p = np.ones((self.Nx, self.Ny))

    def prim_to_conservative(self):
        """Convert primitive to conservative"""
        self.Q[0] = self.rho
        self.Q[1] = self.rho * self.u
        self.Q[2] = self.rho * self.v
        kinetic = 0.5 * self.rho * (self.u**2 + self.v**2)
        self.Q[3] = self.p / (self.gamma - 1.0) + kinetic

    def conservative_to_primitive(self):
        """Convert conservative to primitive"""
        np.maximum(self.Q[0], 1e-10, out=self.rho)
        self.u = self.Q[1] / self.rho
        self.v = self.Q[2] / self.rho

        kinetic = 0.5 * self.rho * (self.u**2 + self.v**2)
        e_int = self.Q[3] / self.rho - kinetic
        np.maximum(e_int, 1e-10, out=e_int)
        self.p = (self.gamma - 1.0) * self.rho * e_int
        np.maximum(self.p, 1e-10, out=self.p)

    def get_mach_number(self):
        """Compute Mach number"""
        a = np.sqrt(self.gamma * self.p / (self.rho + 1e-14))
        speed = np.sqrt(self.u**2 + self.v**2)
        return speed / (a + 1e-14)

# Global state
state = FlowState()

Overwriting state.py


## Flux

In [ ]:
%%writefile flux.py
"""
Physical flux functions for Euler equations.
"""

import numpy as np
from parameters import params

def flux_x(Q, gamma=params.gamma):
    """Flux in x-direction"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * rho * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)

    F = np.zeros_like(Q)
    F[0] = Q[1]
    F[1] = Q[1] * u + p
    F[2] = Q[1] * v
    F[3] = u * (Q[3] + p)
    return F

def flux_y(Q, gamma=params.gamma):
    """Flux in y-direction"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * rho * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)

    G = np.zeros_like(Q)
    G[0] = Q[2]
    G[1] = Q[2] * u
    G[2] = Q[2] * v + p
    G[3] = v * (Q[3] + p)
    return G

Overwriting flux.py


## Reconstruct

In [ ]:
%%writefile reconstruction.py
"""
MUSCL reconstruction with minmod limiter.
"""

import numpy as np

def minmod(a, b):
    """Minmod limiter"""
    return 0.5 * (np.sign(a) + np.sign(b)) * np.minimum(np.abs(a), np.abs(b))

def apply_bc_x(Q):
    """Transmissive BCs in x"""
    Q[:, 0, :] = Q[:, 1, :]
    Q[:, -1, :] = Q[:, -2, :]
    return Q

def apply_bc_y(Q):
    """Periodic BCs in y"""
    Q[:, :, 0] = Q[:, :, -2]
    Q[:, :, -1] = Q[:, :, 1]
    return Q

def reconstruct_x(Q):
    """MUSCL reconstruction in x"""
    Nx, Ny = Q.shape[1], Q.shape[2]

    Q_bc = apply_bc_x(Q.copy())
    Qg = np.zeros((4, Nx+2, Ny))
    Qg[:, 1:Nx+1, :] = Q_bc
    Qg[:, 0, :] = Q_bc[:, 0, :]
    Qg[:, -1, :] = Q_bc[:, -1, :]

    dQ_left = Qg[:, 1:-1, :] - Qg[:, :-2, :]
    dQ_right = Qg[:, 2:, :] - Qg[:, 1:-1, :]
    slope = minmod(dQ_left, dQ_right)

    Q_face_right = Q + 0.5 * slope
    Q_face_left = Q - 0.5 * slope

    QL = np.zeros((4, Nx+1, Ny))
    QR = np.zeros((4, Nx+1, Ny))

    QL[:, 1:Nx, :] = Q_face_right[:, :-1, :]
    QR[:, 1:Nx, :] = Q_face_left[:, 1:, :]
    QL[:, 0, :] = Q_face_left[:, 0, :]
    QR[:, 0, :] = Q_face_right[:, 0, :]
    QL[:, Nx, :] = Q_face_right[:, -1, :]
    QR[:, Nx, :] = Q_face_left[:, -1, :]

    return QL, QR

def reconstruct_y(Q):
    """MUSCL reconstruction in y"""
    Nx, Ny = Q.shape[1], Q.shape[2]

    Q_bc = apply_bc_y(Q.copy())
    Qg = np.zeros((4, Nx, Ny+2))
    Qg[:, :, 1:Ny+1] = Q_bc
    Qg[:, :, 0] = Q_bc[:, :, 0]
    Qg[:, :, -1] = Q_bc[:, :, -1]

    dQ_left = Qg[:, :, 1:-1] - Qg[:, :, :-2]
    dQ_right = Qg[:, :, 2:] - Qg[:, :, 1:-1]
    slope = minmod(dQ_left, dQ_right)

    Q_face_right = Q + 0.5 * slope
    Q_face_left = Q - 0.5 * slope

    QL = np.zeros((4, Nx, Ny+1))
    QR = np.zeros((4, Nx, Ny+1))

    QL[:, :, 1:Ny] = Q_face_right[:, :, :-1]
    QR[:, :, 1:Ny] = Q_face_left[:, :, 1:]
    QL[:, :, 0] = Q_face_left[:, :, 0]
    QR[:, :, 0] = Q_face_right[:, :, 0]
    QL[:, :, Ny] = Q_face_right[:, :, -1]
    QR[:, :, Ny] = Q_face_left[:, :, -1]

    return QL, QR

Overwriting reconstruction.py


## WENO applied here

In [ ]:
%%writefile weno5.py
"""
Optimized 5th-order WENO reconstruction.
Uses numpy vectorization for speed.
"""

import numpy as np

def weno5_reconstruct_x(Q):
    """
    Optimized WENO5 reconstruction in x-direction.
    """
    Nx, Ny = Q.shape[1], Q.shape[2]
    eps = 1e-6

    # Apply BCs (need 3 ghost points)
    from reconstruction import apply_bc_x
    Q_bc = apply_bc_x(Q.copy())

    # Add ghost cells (3 on each side)
    Qg = np.zeros((4, Nx+6, Ny))
    Qg[:, 3:3+Nx, :] = Q_bc

    # Fill ghosts efficiently
    for i in range(3):
        Qg[:, 2-i, :] = Q_bc[:, i, :]
        Qg[:, 3+Nx+i, :] = Q_bc[:, -1-i, :]

    # Pre-allocate
    QL = np.zeros((4, Nx+1, Ny))

    # WENO5 weights (constant)
    c00, c01, c02 = 1.0/3.0, -7.0/6.0, 11.0/6.0
    c10, c11, c12 = -1.0/6.0, 5.0/6.0, 1.0/3.0
    c20, c21, c22 = 1.0/3.0, 5.0/6.0, -1.0/6.0

    # Vectorized WENO for each variable
    for n in range(4):
        for i in range(Nx+1):
            # Make sure indices are within bounds
            idx = i + 2
            if idx + 4 < Qg.shape[1]:  # Check bounds
                # Get stencil (vectorized over y)
                q0 = Qg[n, idx, :]
                q1 = Qg[n, idx+1, :]
                q2 = Qg[n, idx+2, :]
                q3 = Qg[n, idx+3, :]
                q4 = Qg[n, idx+4, :]

                # Candidate stencils
                q0_stencil = c00*q0 + c01*q1 + c02*q2
                q1_stencil = c10*q1 + c11*q2 + c12*q3
                q2_stencil = c20*q2 + c21*q3 + c22*q4

                # Smoothness indicators
                beta0 = 13.0/12.0 * (q0 - 2*q1 + q2)**2 + 0.25 * (q0 - 4*q1 + 3*q2)**2
                beta1 = 13.0/12.0 * (q1 - 2*q2 + q3)**2 + 0.25 * (q1 - q3)**2
                beta2 = 13.0/12.0 * (q2 - 2*q3 + q4)**2 + 0.25 * (3*q2 - 4*q3 + q4)**2

                # WENO weights
                alpha0 = 0.1 / (beta0 + eps)**2
                alpha1 = 0.6 / (beta1 + eps)**2
                alpha2 = 0.3 / (beta2 + eps)**2

                alpha_sum = alpha0 + alpha1 + alpha2

                w0 = alpha0 / (alpha_sum + 1e-14)
                w1 = alpha1 / (alpha_sum + 1e-14)
                w2 = alpha2 / (alpha_sum + 1e-14)

                QL[n, i, :] = w0*q0_stencil + w1*q1_stencil + w2*q2_stencil
            else:
                # Fallback to simple extrapolation at boundaries
                QL[n, i, :] = Q_bc[n, min(i, Nx-1), :]

    # Right states
    QR = np.zeros((4, Nx+1, Ny))

    for n in range(4):
        for i in range(Nx+1):
            idx = i + 3
            if idx + 4 < Qg.shape[1]:
                q1 = Qg[n, idx, :]
                q2 = Qg[n, idx+1, :]
                q3 = Qg[n, idx+2, :]
                q4 = Qg[n, idx+3, :]
                q5 = Qg[n, idx+4, :]

                # Mirrored stencils
                q0_stencil = c22*q1 + c21*q2 + c20*q3
                q1_stencil = c12*q2 + c11*q3 + c10*q4
                q2_stencil = c02*q3 + c01*q4 + c00*q5

                beta0 = 13.0/12.0 * (q1 - 2*q2 + q3)**2 + 0.25 * (q1 - 4*q2 + 3*q3)**2
                beta1 = 13.0/12.0 * (q2 - 2*q3 + q4)**2 + 0.25 * (q2 - q4)**2
                beta2 = 13.0/12.0 * (q3 - 2*q4 + q5)**2 + 0.25 * (3*q3 - 4*q4 + q5)**2

                alpha0 = 0.1 / (beta0 + eps)**2
                alpha1 = 0.6 / (beta1 + eps)**2
                alpha2 = 0.3 / (beta2 + eps)**2

                alpha_sum = alpha0 + alpha1 + alpha2

                w0 = alpha0 / (alpha_sum + 1e-14)
                w1 = alpha1 / (alpha_sum + 1e-14)
                w2 = alpha2 / (alpha_sum + 1e-14)

                QR[n, i, :] = w0*q0_stencil + w1*q1_stencil + w2*q2_stencil
            else:
                QR[n, i, :] = Q_bc[n, min(i, Nx-1), :]

    return QL, QR

def weno5_reconstruct_y(Q):
    """WENO5 in y-direction (similar optimization)"""
    Nx, Ny = Q.shape[1], Q.shape[2]
    eps = 1e-6

    from reconstruction import apply_bc_y
    Q_bc = apply_bc_y(Q.copy())

    Qg = np.zeros((4, Nx, Ny+6))
    Qg[:, :, 3:3+Ny] = Q_bc

    for j in range(3):
        Qg[:, :, 2-j] = Q_bc[:, :, j]
        Qg[:, :, 3+Ny+j] = Q_bc[:, :, -1-j]

    QL = np.zeros((4, Nx, Ny+1))

    for n in range(4):
        for j in range(Ny+1):
            idx = j + 2
            if idx + 4 < Qg.shape[2]:
                q0 = Qg[n, :, idx]
                q1 = Qg[n, :, idx+1]
                q2 = Qg[n, :, idx+2]
                q3 = Qg[n, :, idx+3]
                q4 = Qg[n, :, idx+4]

                q0_stencil = (1.0/3.0)*q0 - (7.0/6.0)*q1 + (11.0/6.0)*q2
                q1_stencil = -(1.0/6.0)*q1 + (5.0/6.0)*q2 + (1.0/3.0)*q3
                q2_stencil = (1.0/3.0)*q2 + (5.0/6.0)*q3 - (1.0/6.0)*q4

                beta0 = 13.0/12.0 * (q0 - 2*q1 + q2)**2 + 0.25 * (q0 - 4*q1 + 3*q2)**2
                beta1 = 13.0/12.0 * (q1 - 2*q2 + q3)**2 + 0.25 * (q1 - q3)**2
                beta2 = 13.0/12.0 * (q2 - 2*q3 + q4)**2 + 0.25 * (3*q2 - 4*q3 + q4)**2

                alpha0 = 0.1 / (beta0 + eps)**2
                alpha1 = 0.6 / (beta1 + eps)**2
                alpha2 = 0.3 / (beta2 + eps)**2

                alpha_sum = alpha0 + alpha1 + alpha2

                w0 = alpha0 / (alpha_sum + 1e-14)
                w1 = alpha1 / (alpha_sum + 1e-14)
                w2 = alpha2 / (alpha_sum + 1e-14)

                QL[n, :, j] = w0*q0_stencil + w1*q1_stencil + w2*q2_stencil
            else:
                QL[n, :, j] = Q_bc[n, :, min(j, Ny-1)]

    # Similar for QR...
    QR = np.zeros((4, Nx, Ny+1))

    for n in range(4):
        for j in range(Ny+1):
            idx = j + 3
            if idx + 4 < Qg.shape[2]:
                q1 = Qg[n, :, idx]
                q2 = Qg[n, :, idx+1]
                q3 = Qg[n, :, idx+2]
                q4 = Qg[n, :, idx+3]
                q5 = Qg[n, :, idx+4]

                q0_stencil = (1.0/3.0)*q1 + (5.0/6.0)*q2 - (1.0/6.0)*q3
                q1_stencil = -(1.0/6.0)*q2 + (5.0/6.0)*q3 + (1.0/3.0)*q4
                q2_stencil = (1.0/3.0)*q3 - (7.0/6.0)*q4 + (11.0/6.0)*q5

                beta0 = 13.0/12.0 * (q1 - 2*q2 + q3)**2 + 0.25 * (q1 - 4*q2 + 3*q3)**2
                beta1 = 13.0/12.0 * (q2 - 2*q3 + q4)**2 + 0.25 * (q2 - q4)**2
                beta2 = 13.0/12.0 * (q3 - 2*q4 + q5)**2 + 0.25 * (3*q3 - 4*q4 + q5)**2

                alpha0 = 0.1 / (beta0 + eps)**2
                alpha1 = 0.6 / (beta1 + eps)**2
                alpha2 = 0.3 / (beta2 + eps)**2

                alpha_sum = alpha0 + alpha1 + alpha2

                w0 = alpha0 / (alpha_sum + 1e-14)
                w1 = alpha1 / (alpha_sum + 1e-14)
                w2 = alpha2 / (alpha_sum + 1e-14)

                QR[n, :, j] = w0*q0_stencil + w1*q1_stencil + w2*q2_stencil
            else:
                QR[n, :, j] = Q_bc[n, :, min(j, Ny-1)]

    return QL, QR

Overwriting weno5.py


## Riemann

In [ ]:
%%writefile riemann.py
"""
HLLC Riemann solver.
"""

import numpy as np

def hllc_flux_x(QL, QR, gamma=1.4):
    """HLLC flux in x-direction"""
    rhoL = np.maximum(QL[0], 1e-10)
    uL = QL[1] / rhoL
    vL = QL[2] / rhoL
    pL = np.maximum((gamma - 1.0) * (QL[3] - 0.5 * rhoL * (uL**2 + vL**2)), 1e-10)
    aL = np.sqrt(gamma * pL / rhoL)

    rhoR = np.maximum(QR[0], 1e-10)
    uR = QR[1] / rhoR
    vR = QR[2] / rhoR
    pR = np.maximum((gamma - 1.0) * (QR[3] - 0.5 * rhoR * (uR**2 + vR**2)), 1e-10)
    aR = np.sqrt(gamma * pR / rhoR)

    SL = np.minimum(uL - aL, uR - aR)
    SR = np.maximum(uL + aL, uR + aR)

    num = pR - pL + rhoL * uL * (SL - uL) - rhoR * uR * (SR - uR)
    denom = rhoL * (SL - uL) - rhoR * (SR - uR) + 1e-14
    S_star = num / denom

    FL = np.zeros_like(QL)
    FL[0] = rhoL * uL
    FL[1] = rhoL * uL * uL + pL
    FL[2] = rhoL * uL * vL
    FL[3] = uL * (QL[3] + pL)

    FR = np.zeros_like(QR)
    FR[0] = rhoR * uR
    FR[1] = rhoR * uR * uR + pR
    FR[2] = rhoR * uR * vR
    FR[3] = uR * (QR[3] + pR)

    F = np.zeros_like(FL)
    mask1 = SL >= 0
    mask2 = (SL < 0) & (S_star >= 0)
    mask3 = (S_star < 0) & (SR >= 0)

    for i in range(4):
        F[i] = np.where(mask1, FL[i],
               np.where(mask2 | mask3,
                       (SR * FL[i] - SL * FR[i] + SL * SR * (QR[i] - QL[i])) / (SR - SL + 1e-14),
                       FR[i]))
    return F

def hllc_flux_y(QL, QR, gamma=1.4):
    """HLLC flux in y-direction"""
    QL_swapped = QL.copy()
    QR_swapped = QR.copy()
    QL_swapped[1] = QL[2]
    QL_swapped[2] = QL[1]
    QR_swapped[1] = QR[2]
    QR_swapped[2] = QR[1]

    F_swapped = hllc_flux_x(QL_swapped, QR_swapped, gamma)
    F = F_swapped.copy()
    F[1] = F_swapped[2]
    F[2] = F_swapped[1]
    return F

Overwriting riemann.py


## Flux-divergence

In [ ]:
%%writefile flux_divergence.py
"""
Compute flux divergence.
"""

import numpy as np
from parameters import params
from reconstruction import reconstruct_x, reconstruct_y
from riemann import hllc_flux_x, hllc_flux_y

def compute_rhs(Q, dx, dy):
    """Compute RHS = -(dF/dx + dG/dy)"""
    Nx, Ny = Q.shape[1], Q.shape[2]

    QL_x, QR_x = reconstruct_x(Q)
    QL_y, QR_y = reconstruct_y(Q)

    Fx = hllc_flux_x(QL_x, QR_x, params.gamma)
    Fy = hllc_flux_y(QL_y, QR_y, params.gamma)

    dFx = (Fx[:, 1:Nx+1, :] - Fx[:, 0:Nx, :]) / dx
    dFy = (Fy[:, :, 1:Ny+1] - Fy[:, :, 0:Ny]) / dy

    return -(dFx + dFy)

Overwriting flux_divergence.py


## Timestep

In [ ]:
%%writefile timestep.py
"""
CFL condition and time step.
"""

import numpy as np
from parameters import params

def compute_dt(Q, dx, dy):
    """Compute stable time step"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (params.gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)
    a = np.sqrt(params.gamma * p / (rho + 1e-14))

    max_speed_x = np.max(np.abs(u) + a)
    max_speed_y = np.max(np.abs(v) + a)
    max_speed_x = max(max_speed_x, 1e-10)
    max_speed_y = max(max_speed_y, 1e-10)

    dt = params.CFL * min(dx / max_speed_x, dy / max_speed_y)
    return dt

Overwriting timestep.py


## TVD Runge-Kutta 3 times integration

In [ ]:
%%writefile rk3.py
"""
TVD Runge-Kutta 3 time integration.
"""

import numpy as np
from flux_divergence import compute_rhs
from reconstruction import apply_bc_x, apply_bc_y

def apply_boundary_conditions(Q):
    """Apply all boundary conditions"""
    Q = apply_bc_x(Q)
    Q = apply_bc_y(Q)
    return Q

def rk3_step(Q, dt, dx, dy):
    """Take one RK3 time step"""
    # Stage 1
    L0 = compute_rhs(Q, dx, dy)
    Q1 = Q + dt * L0
    Q1 = apply_boundary_conditions(Q1)
    Q1 = np.nan_to_num(Q1, nan=1e-10)
    Q1[0] = np.maximum(Q1[0], 1e-10)
    Q1[3] = np.maximum(Q1[3], 1e-10)

    # Stage 2
    L1 = compute_rhs(Q1, dx, dy)
    Q2 = 0.75 * Q + 0.25 * (Q1 + dt * L1)
    Q2 = apply_boundary_conditions(Q2)
    Q2 = np.nan_to_num(Q2, nan=1e-10)
    Q2[0] = np.maximum(Q2[0], 1e-10)
    Q2[3] = np.maximum(Q2[3], 1e-10)

    # Stage 3
    L2 = compute_rhs(Q2, dx, dy)
    Q3 = (1.0/3.0) * Q + (2.0/3.0) * (Q2 + dt * L2)
    Q3 = apply_boundary_conditions(Q3)
    Q3 = np.nan_to_num(Q3, nan=1e-10)
    Q3[0] = np.maximum(Q3[0], 1e-10)
    Q3[3] = np.maximum(Q3[3], 1e-10)

    return Q3

Overwriting rk3.py


## Initial Conditions

In [ ]:
%%writefile initial_conditions.py
"""
Initial conditions for Sod shock tube.
"""

import numpy as np
import mesh
import state

def init_sod():
    """Initialize Sod shock tube"""
    X = mesh.mesh.X
    x_diaphragm = mesh.mesh.Lx / 2.0

    state.state.rho = np.where(X < x_diaphragm, 1.0, 0.125)
    state.state.u = np.zeros_like(X)
    state.state.v = np.zeros_like(X)
    state.state.p = np.where(X < x_diaphragm, 1.0, 0.1)

    state.state.prim_to_conservative()

    print("Sod shock tube initialized")
    print(f"  Diaphragm at x = {x_diaphragm:.2f}")
    print(f"  Left:  rho=1.0, p=1.0")
    print(f"  Right: rho=0.125, p=0.1")

Overwriting initial_conditions.py


## Exact_sod


In [ ]:
%%writefile exact_sod.py
"""
Exact solution for Sod shock tube.
"""

import numpy as np
from parameters import params

def exact_sod_solution(x, t, gamma=params.gamma, x_diaphragm=0.5):
    """Exact solution for Sod shock tube"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)

    def fL(p):
        if p > pL:
            A = 2.0 / ((gamma + 1.0) * rhoL)
            B = (gamma - 1.0) / (gamma + 1.0) * pL
            return (p - pL) * np.sqrt(A / (p + B + 1e-14))
        else:
            return (2.0 * aL / (gamma - 1.0)) * ((p / pL)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def fR(p):
        if p > pR:
            A = 2.0 / ((gamma + 1.0) * rhoR)
            B = (gamma - 1.0) / (gamma + 1.0) * pR
            return (p - pR) * np.sqrt(A / (p + B + 1e-14))
        else:
            return (2.0 * aR / (gamma - 1.0)) * ((p / pR)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def f(p):
        return fL(p) + fR(p) + (uR - uL)

    def df(p):
        dp = max(1e-6 * p, 1e-8)
        return (f(p + dp) - f(p - dp)) / (2.0 * dp)

    p_star = 0.5 * (pL + pR)
    for _ in range(50):
        dp = -f(p_star) / (df(p_star) + 1e-14)
        p_star += dp
        if abs(dp) < 1e-12:
            break

    u_star = 0.5 * (uL + uR) + 0.5 * (fR(p_star) - fL(p_star))

    aL_star = aL * (p_star / pL)**((gamma - 1.0) / (2.0 * gamma))
    S_HL = uL - aL
    S_TL = u_star - aL_star
    S_contact = u_star
    S_R = uR + aR * np.sqrt((gamma + 1.0) / (2.0 * gamma) * (p_star / pR) +
                            (gamma - 1.0) / (2.0 * gamma))

    rhoL_star = rhoL * (p_star / pL)**(1.0 / gamma)
    rhoR_star = rhoR * ((p_star / pR + (gamma - 1.0) / (gamma + 1.0)) /
                        ((gamma - 1.0) / (gamma + 1.0) * p_star / pR + 1.0))

    if t <= 0:
        return (np.where(x < x_diaphragm, rhoL, rhoR),
                np.where(x < x_diaphragm, uL, uR),
                np.where(x < x_diaphragm, pL, pR))

    xi = (x - x_diaphragm) / (t + 1e-14)
    rho = np.zeros_like(x)
    u = np.zeros_like(x)
    p = np.zeros_like(x)

    for i, s in enumerate(xi):
        if s <= S_HL:
            rho[i], u[i], p[i] = rhoL, uL, pL
        elif s <= S_TL:
            u_tmp = 2.0 / (gamma + 1.0) * (aL + (gamma - 1.0) / 2.0 * uL + s)
            a_tmp = aL + (gamma - 1.0) / 2.0 * (uL - u_tmp)
            rho[i] = rhoL * (a_tmp / aL)**(2.0 / (gamma - 1.0))
            u[i] = u_tmp
            p[i] = pL * (a_tmp / aL)**(2.0 * gamma / (gamma - 1.0))
        elif s <= S_contact:
            rho[i], u[i], p[i] = rhoL_star, u_star, p_star
        elif s <= S_R:
            rho[i], u[i], p[i] = rhoR_star, u_star, p_star
        else:
            rho[i], u[i], p[i] = rhoR, uR, pR

    return rho, u, p

Overwriting exact_sod.py


## Plotting

In [ ]:
%%writefile plotting.py
"""
Plotting functions.
"""

import numpy as np
import matplotlib.pyplot as plt
import os
from parameters import params
import mesh
import state
from exact_sod import exact_sod_solution

def ensure_output_dir():
    """Ensure output directory exists"""
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

def plot_verification(t, step):
    """Plot numerical vs exact solution"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    p = state.state.p

    x = mesh.mesh.x_centers
    rho_num = rho.mean(axis=1)
    u_num = u.mean(axis=1)
    p_num = p.mean(axis=1)

    x_diaphragm = mesh.mesh.Lx / 2.0
    rho_ex, u_ex, p_ex = exact_sod_solution(x, t, x_diaphragm=x_diaphragm)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Sod Shock Tube - t={t:.4f}, step={step}", fontsize=14)

    # Density
    axes[0].plot(x, rho_ex, 'k-', lw=2, label='Exact')
    axes[0].plot(x, rho_num, 'ro--', ms=3, lw=1, alpha=0.7, label='MUSCL-HLLC')
    axes[0].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([0, mesh.mesh.Lx])

    # Velocity
    axes[1].plot(x, u_ex, 'k-', lw=2, label='Exact')
    axes[1].plot(x, u_num, 'go--', ms=3, lw=1, alpha=0.7, label='MUSCL-HLLC')
    axes[1].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('Velocity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([0, mesh.mesh.Lx])

    # Pressure
    axes[2].plot(x, p_ex, 'k-', lw=2, label='Exact')
    axes[2].plot(x, p_num, 'bo--', ms=3, lw=1, alpha=0.7, label='MUSCL-HLLC')
    axes[2].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Pressure')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim([0, mesh.mesh.Lx])

    plt.tight_layout()
    filename = f"{params.output_dir}/plots/verification_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Verification plot saved: {filename}")

def plot_schlieren(t, step, k=10.0):
    """Plot numerical schlieren"""
    ensure_output_dir()

    rho = state.state.rho
    drho_dx = np.gradient(rho, mesh.mesh.dx, axis=0)
    drho_dy = np.gradient(rho, mesh.mesh.dy, axis=1)
    grad_mag = np.sqrt(drho_dx**2 + drho_dy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-14)
    schlieren = np.exp(-k * grad_norm)

    fig, ax = plt.subplots(figsize=(12, 3))
    im = ax.imshow(schlieren.T, origin='lower',
                   extent=[0, mesh.mesh.Lx, 0, mesh.mesh.Ly],
                   cmap='gray', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Schlieren |∇ρ| - t={t:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    filename = f"{params.output_dir}/plots/schlieren_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Schlieren plot saved: {filename}")

def plot_mach(t, step):
    """Plot Mach number"""
    ensure_output_dir()

    M = state.state.get_mach_number()

    fig, ax = plt.subplots(figsize=(12, 3))
    cf = ax.contourf(mesh.mesh.X, mesh.mesh.Y, M, levels=40, cmap='jet')
    plt.colorbar(cf, ax=ax, label='Mach Number')
    ax.contour(mesh.mesh.X, mesh.mesh.Y, M, levels=[1.0], colors='white',
               linewidths=1.5, linestyles='--')
    ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Mach Number - t={t:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    filename = f"{params.output_dir}/plots/mach_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Mach plot saved: {filename}")

def plot_contours(t, step):
    """Plot 2D contours"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    v = state.state.v
    p = state.state.p
    M = state.state.get_mach_number()
    T = p / (rho * params.R)

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle(f"Flow Fields - t={t:.4f}", fontsize=14)

    fields = [
        (axes[0,0], rho, 'Density', 'viridis'),
        (axes[0,1], u, 'Velocity u', 'RdBu_r'),
        (axes[0,2], v, 'Velocity v', 'RdBu_r'),
        (axes[1,0], p, 'Pressure', 'plasma'),
        (axes[1,1], T, 'Temperature', 'hot'),
        (axes[1,2], M, 'Mach Number', 'jet')
    ]

    for ax, field, title, cmap in fields:
        cf = ax.contourf(mesh.mesh.X, mesh.mesh.Y, field, levels=40, cmap=cmap)
        plt.colorbar(cf, ax=ax)
        ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    plt.tight_layout()
    filename = f"{params.output_dir}/plots/contours_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Contours plot saved: {filename}")

Overwriting plotting.py


## saving

In [ ]:
%%writefile saving.py
"""
Saving functions for data and VTK files.
"""

import numpy as np
import matplotlib.pyplot as plt  # Add this import
import os
from parameters import params
import mesh
import state
from plotting import plot_verification, plot_schlieren, plot_mach, plot_contours

def ensure_output_dir():
    """Ensure output directories exist"""
    os.makedirs(f"{params.output_dir}/data", exist_ok=True)
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)
    os.makedirs(f"{params.output_dir}/vtk", exist_ok=True)

def save_data(t, step, prefix=""):
    """Save flow field data to numpy files"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    v = state.state.v
    p = state.state.p
    M = state.state.get_mach_number()

    # 1D profiles (averaged over y)
    x1d = mesh.mesh.x_centers
    rho1d = rho.mean(axis=1)
    u1d = u.mean(axis=1)
    p1d = p.mean(axis=1)

    # Save 1D profiles
    if prefix:
        filename = f"{params.output_dir}/data/{prefix}_data_t{t:.4f}.npz"
    else:
        filename = f"{params.output_dir}/data/data_t{t:.4f}_step{step:06d}.npz"

    np.savez(filename,
             x=x1d, rho=rho1d, u=u1d, p=p1d,
             t=t, step=step, Lx=mesh.mesh.Lx)

    # Save 2D fields
    if prefix:
        filename_2d = f"{params.output_dir}/data/{prefix}_fields_t{t:.4f}.npz"
    else:
        filename_2d = f"{params.output_dir}/data/fields_t{t:.4f}_step{step:06d}.npz"

    np.savez(filename_2d,
             X=mesh.mesh.X, Y=mesh.mesh.Y,
             rho=rho, u=u, v=v, p=p, M=M,
             t=t, step=step, Lx=mesh.mesh.Lx)

    print(f"  Data saved: {filename}")

def save_vtk(t, step, prefix=""):
    """
    Save data in VTK format for ParaView visualization.
    """
    ensure_output_dir()

    if prefix:
        filename = f"{params.output_dir}/vtk/{prefix}_solution_t{t:.4f}.vtk"
    else:
        filename = f"{params.output_dir}/vtk/solution_t{t:.4f}_step{step:06d}.vtk"

    with open(filename, 'w') as f:
        # Write VTK header
        f.write("# vtk DataFile Version 3.0\n")
        f.write(f"Sod shock tube solution at t={t}\n")
        f.write("ASCII\n")
        f.write("DATASET STRUCTURED_GRID\n")
        f.write(f"DIMENSIONS {mesh.mesh.Nx} {mesh.mesh.Ny} 1\n")
        f.write(f"POINTS {mesh.mesh.Nx * mesh.mesh.Ny} float\n")

        # Write grid points
        for j in range(mesh.mesh.Ny):
            for i in range(mesh.mesh.Nx):
                f.write(f"{mesh.mesh.X[i,j]} {mesh.mesh.Y[i,j]} 0.0\n")

        # Write point data
        f.write(f"POINT_DATA {mesh.mesh.Nx * mesh.mesh.Ny}\n")

        # Density
        f.write("SCALARS density float 1\n")
        f.write("LOOKUP_TABLE default\n")
        for j in range(mesh.mesh.Ny):
            for i in range(mesh.mesh.Nx):
                f.write(f"{state.state.rho[i,j]}\n")

        # Pressure
        f.write("SCALARS pressure float 1\n")
        f.write("LOOKUP_TABLE default\n")
        for j in range(mesh.mesh.Ny):
            for i in range(mesh.mesh.Nx):
                f.write(f"{state.state.p[i,j]}\n")

        # Velocity
        f.write("VECTORS velocity float\n")
        for j in range(mesh.mesh.Ny):
            for i in range(mesh.mesh.Nx):
                f.write(f"{state.state.u[i,j]} {state.state.v[i,j]} 0.0\n")

        # Mach number
        f.write("SCALARS mach float 1\n")
        f.write("LOOKUP_TABLE default\n")
        M = state.state.get_mach_number()
        for j in range(mesh.mesh.Ny):
            for i in range(mesh.mesh.Nx):
                f.write(f"{M[i,j]}\n")

        # Temperature
        f.write("SCALARS temperature float 1\n")
        f.write("LOOKUP_TABLE default\n")
        for j in range(mesh.mesh.Ny):
            for i in range(mesh.mesh.Nx):
                T = state.state.p[i,j] / (state.state.rho[i,j] * params.R)
                f.write(f"{T}\n")

    print(f"  VTK file saved: {filename}")

def save_all_time_step(t, step):
    """Save all data and plots for a time step"""
    print(f"\n--- Saving outputs for t={t:.4f}, step={step} ---")
    save_data(t, step)
    save_vtk(t, step)
    plot_verification(t, step)
    plot_schlieren(t, step)
    plot_mach(t, step)
    plot_contours(t, step)
    print(f"--- Completed t={t:.4f} ---\n")

def plot_energy_history(history):
    """Plot energy conservation over time"""
    ensure_output_dir()

    times = [h[0] for h in history]
    E_tots = [h[1][3].sum() * mesh.mesh.dx * mesh.mesh.dy for h in history]

    E0 = E_tots[0]
    E_rel = [(e - E0) / E0 * 100 for e in E_tots]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Energy Conservation', fontsize=14)

    axes[0].plot(times, E_tots, 'b-o', ms=4)
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Total Energy')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title('Total Energy')

    axes[1].plot(times, E_rel, 'r-o', ms=4)
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Relative Error (%)')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_title('Energy Conservation Error')

    plt.tight_layout()
    fname = f"{params.output_dir}/plots/energy_conservation.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Energy plot saved: {fname}")

Overwriting saving.py


## Benchmarking

In [ ]:
%%writefile benchmark.py
"""
Performance benchmarking tool for MUSCL vs WENO5.
"""

import numpy as np
import time
import matplotlib.pyplot as plt
from parameters import params
import state
from initial_conditions import init_sod
from flux_divergence import compute_rhs
from timestep import compute_dt
from rk3 import rk3_step

class SolverBenchmark:
    def __init__(self):
        self.resolutions = [100, 200]  # Grid sizes to test
        self.n_steps = 500  # Steps per benchmark
        self.results = {}

    def run_benchmark(self, scheme):
        """Run benchmark for a given scheme"""
        params.reconstruction_scheme = scheme

        # Test different resolutions
        scheme_results = {
            'resolutions': [],
            'steps_per_sec': [],
            'grid_points': [],
            'memory_mb': []
        }

        for Nx in self.resolutions:
            # Set resolution
            Ny = Nx  # Square grid
            params.Nx = Nx
            params.Ny = Ny
            params.update_grid()

            # Reinitialize state
            import mesh
            mesh.mesh = mesh.Mesh()
            import state
            state.state = state.FlowState()

            # Initialize flow
            init_sod()

            # Warm-up steps
            for _ in range(10):
                dt = compute_dt(state.state.Q, params.dx, params.dy)
                state.state.Q = rk3_step(state.state.Q, dt, params.dx, params.dy)
                state.state.conservative_to_primitive()

            # Benchmark
            start_time = time.time()
            for step in range(self.n_steps):
                dt = compute_dt(state.state.Q, params.dx, params.dy)
                state.state.Q = rk3_step(state.state.Q, dt, params.dx, params.dy)
                state.state.conservative_to_primitive()
            elapsed = time.time() - start_time

            # Calculate metrics
            grid_points = Nx * Ny
            steps_per_sec = self.n_steps / elapsed
            memory_mb = state.state.Q.nbytes / (1024 * 1024)

            scheme_results['resolutions'].append(Nx)
            scheme_results['steps_per_sec'].append(steps_per_sec)
            scheme_results['grid_points'].append(grid_points)
            scheme_results['memory_mb'].append(memory_mb)

            print(f"  {scheme} - N={Nx}: {steps_per_sec:.2f} steps/s, {memory_mb:.2f} MB")

        return scheme_results

    def compare(self):
        """Compare MUSCL vs WENO5"""
        print("\n" + "="*70)
        print("BENCHMARKING: MUSCL vs WENO5")
        print("="*70)

        # Run benchmarks
        print("\nRunning MUSCL benchmarks...")
        muscl_results = self.run_benchmark("MUSCL")

        print("\nRunning WENO5 benchmarks...")
        weno5_results = self.run_benchmark("WENO5")

        # Store results
        self.results = {
            'MUSCL': muscl_results,
            'WENO5': weno5_results
        }

        # Print comparison table
        print("\n" + "="*70)
        print("PERFORMANCE COMPARISON")
        print("="*70)
        print(f"{'Resolution':<12} {'MUSCL (steps/s)':<18} {'WENO5 (steps/s)':<18} {'Speedup':<12}")
        print("-"*70)

        for i, Nx in enumerate(self.resolutions):
            muscl_speed = muscl_results['steps_per_sec'][i]
            weno5_speed = weno5_results['steps_per_sec'][i]
            speedup = muscl_speed / weno5_speed
            print(f"{Nx:>4}x{Nx:<6} {muscl_speed:<18.2f} {weno5_speed:<18.2f} {speedup:<12.2f}x")

        # Plot results
        self.plot_results()

        return self.results

    def plot_results(self):
        """Generate performance plots"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Plot 1: Steps per second vs Grid points
        ax = axes[0]
        for scheme, results in self.results.items():
            ax.plot(results['grid_points'], results['steps_per_sec'],
                   'o-', label=scheme, linewidth=2, markersize=8)

        ax.set_xlabel('Grid Points')
        ax.set_ylabel('Steps per Second')
        ax.set_title('Computational Speed')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')

        # Plot 2: Speedup ratio
        ax = axes[1]
        speedups = [m/w for m, w in zip(self.results['MUSCL']['steps_per_sec'],
                                        self.results['WENO5']['steps_per_sec'])]

        ax.plot(self.resolutions, speedups, 'ro-', linewidth=2, markersize=8)
        ax.axhline(y=1.0, color='k', linestyle='--', alpha=0.5, label='Equal')
        ax.set_xlabel('Resolution (Nx)')
        ax.set_ylabel('Speedup (MUSCL / WENO5)')
        ax.set_title('MUSCL Speedup over WENO5')
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('output/performance_comparison.png', dpi=150)
        plt.show()

        # Plot 3: Memory usage
        fig, ax = plt.subplots(figsize=(8, 5))
        for scheme, results in self.results.items():
            ax.plot(results['grid_points'], results['memory_mb'],
                   'o-', label=scheme, linewidth=2, markersize=8)

        ax.set_xlabel('Grid Points')
        ax.set_ylabel('Memory (MB)')
        ax.set_title('Memory Usage')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')

        plt.tight_layout()
        plt.savefig('output/memory_usage.png', dpi=150)
        plt.show()

def run_benchmark():
    """Main benchmarking function"""
    benchmark = SolverBenchmark()
    results = benchmark.compare()

    # Summary statistics
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)

    avg_muscl = np.mean(results['MUSCL']['steps_per_sec'])
    avg_weno5 = np.mean(results['WENO5']['steps_per_sec'])
    avg_speedup = avg_muscl / avg_weno5

    print(f"Average MUSCL speed: {avg_muscl:.2f} steps/s")
    print(f"Average WENO5 speed: {avg_weno5:.2f} steps/s")
    print(f"Average speedup: {avg_speedup:.2f}x")

    # Peak performance
    peak_muscl = max(results['MUSCL']['steps_per_sec'])
    peak_weno5 = max(results['WENO5']['steps_per_sec'])
    print(f"\nPeak MUSCL speed: {peak_muscl:.2f} steps/s")
    print(f"Peak WENO5 speed: {peak_weno5:.2f} steps/s")

    return results

if __name__ == "__main__":
    run_benchmark()

Overwriting benchmark.py


## Main file

In [ ]:
%%writefile main.py
"""
Main solver for Sod shock tube.
"""

import numpy as np
import time
import os
from parameters import params
import mesh
import state
from initial_conditions import init_sod
from timestep import compute_dt
from rk3 import rk3_step
from plotting import plot_verification, plot_schlieren, plot_mach, plot_contours

def run_simulation():
    """Run Sod shock tube simulation"""
    print("\n" + "="*60)
    print("SOD SHOCK TUBE SIMULATION - MUSCL SCHEME")
    print("="*60)
    print(f"Grid: {mesh.mesh.Nx} x {mesh.mesh.Ny}")
    print(f"Domain: x∈[0,{mesh.mesh.Lx}], y∈[0,{mesh.mesh.Ly}]")
    print(f"Final time: {params.t_final}")
    print("="*60)

    # Initialize
    init_sod()
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

    # Plot initial condition
    print("\n" + "="*60)
    print("PLOTTING INITIAL CONDITION (t=0)")
    print("="*60)
    plot_contours(0.0, 0)
    plot_mach(0.0, 0)
    plot_schlieren(0.0, 0)
    plot_verification(0.0, 0)

    # Simulation loop
    t = 0.0
    step = 0
    t_next_save = params.t_print

    print(f"\n{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_mean':>10}  {'p_mean':>10}")
    print("-" * 55)

    while t < params.t_final - 1e-10:
        dt = compute_dt(state.state.Q, mesh.mesh.dx, mesh.mesh.dy)
        dt = min(dt, params.t_final - t)

        state.state.Q = rk3_step(state.state.Q, dt, mesh.mesh.dx, mesh.mesh.dy)
        t += dt
        step += 1
        state.state.conservative_to_primitive()

        if step % 10 == 0:
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  "
                  f"{state.state.rho.mean():>10.4f}  {state.state.p.mean():>10.4f}")

        if t >= t_next_save or abs(t - params.t_final) < 1e-10:
            print(f"\n--- Saving at t={t:.4f}, step={step} ---")
            plot_contours(t, step)
            plot_mach(t, step)
            plot_schlieren(t, step)
            plot_verification(t, step)
            t_next_save += params.t_print
            print("--- Done ---\n")

        if np.any(~np.isfinite(state.state.Q)):
            print(f"NaN detected at step {step}, t={t:.6f}")
            break

    # Plot final condition
    print("\n" + "="*60)
    print("PLOTTING FINAL CONDITION")
    print("="*60)
    plot_contours(t, step)
    plot_mach(t, step)
    plot_schlieren(t, step)
    plot_verification(t, step)
    print("="*60)

    print(f"\nSimulation completed: {step} steps, final time t={t:.5f}")
    return state.state.Q

if __name__ == "__main__":
    start_time = time.time()
    Q_final = run_simulation()
    elapsed = time.time() - start_time
    print(f"\nTotal simulation time: {elapsed:.2f} seconds")
    print(f"Plots saved in: {params.output_dir}/plots/")

Overwriting main.py


## Run the simulation

In [ ]:
# Clear old output and run
!rm -rf output/
!python3 main.py

Grid: 500 x 100
Domain: x∈[0,10.0], y∈[0,1]
dx=0.020000, dy=0.010000

SOD SHOCK TUBE SIMULATION - MUSCL SCHEME
Grid: 500 x 100
Domain: x∈[0,10.0], y∈[0,1]
Final time: 1.0
Sod shock tube initialized
  Diaphragm at x = 5.00
  Left:  rho=1.0, p=1.0
  Right: rho=0.125, p=0.1

PLOTTING INITIAL CONDITION (t=0)
Contours plot saved: output/plots/contours_t0.0000.png
Mach plot saved: output/plots/mach_t0.0000.png
Schlieren plot saved: output/plots/schlieren_t0.0000.png
Verification plot saved: output/plots/verification_t0.0000.png

  Step          t          dt    rho_mean      p_mean
-------------------------------------------------------
    10    0.03381   3.381e-03      0.5625      0.5498

--- Saving at t=0.0505, step=15 ---
Contours plot saved: output/plots/contours_t0.0505.png
Mach plot saved: output/plots/mach_t0.0505.png
Schlieren plot saved: output/plots/schlieren_t0.0505.png
Verification plot saved: output/plots/verification_t0.0505.png
--- Done ---

    20    0.06687   3.260e-03     

In [ ]:
# List all generated files
import os

#print("Data files:")
#!ls -la output/data/ | head -10

print("\nPlot files:")
!ls -la output/plots/ | head -10

#print("\nVTK files:")
#!ls -la output/vtk/ | head -10

Data files:
ls: cannot access 'output/data/': No such file or directory

Plot files:
total 5580
drwxr-xr-x 2 root root   4096 Mar 17 13:11 .
drwxr-xr-x 3 root root   4096 Mar 17 13:10 ..
-rw-r--r-- 1 root root 110228 Mar 17 13:10 contours_t0.0000.png
-rw-r--r-- 1 root root 126751 Mar 17 13:10 contours_t0.0505.png
-rw-r--r-- 1 root root 125545 Mar 17 13:10 contours_t0.1019.png
-rw-r--r-- 1 root root 120761 Mar 17 13:10 contours_t0.1521.png
-rw-r--r-- 1 root root 120739 Mar 17 13:10 contours_t0.2021.png
-rw-r--r-- 1 root root 122302 Mar 17 13:10 contours_t0.2522.png
-rw-r--r-- 1 root root 122897 Mar 17 13:10 contours_t0.3025.png

VTK files:
ls: cannot access 'output/vtk/': No such file or directory
